<a href="https://colab.research.google.com/github/globaltrendarena/global-trend-arena/blob/main/scripts/Competitor_Price_Monitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import xml.etree.ElementTree as ET
import re
import requests
from bs4 import BeautifulSoup
from google.colab import auth
import gspread
from google.auth import default
from urllib.parse import unquote

# ১. গুগল অ্যাকাউন্টে পারমিশন
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# ২. গুগল শিট ওপেন
SITEMAP_URL = "https://inaayasmart.com/product-sitemap.xml"
SHEET_URL = "https://docs.google.com/spreadsheets/d/1h619jwSesr2fnsLnAj4MdSMxMeMSQ3kDzh5Hu56B0r8/edit"

spreadsheet = gc.open_by_url(SHEET_URL)
worksheet = spreadsheet.sheet1

# হেডার চেক ও সেটআপ
existing_data = worksheet.get_all_values()
if not existing_data:
    headers = ["Product Name", "My Product URL", "Competitor URLs", "Min Price", "Max Price", "Average Price"]
    worksheet.append_row(headers)

# ৩. সাইটম্যাপ রিড ও ফিল্টারিং
headers_req = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
response = requests.get(SITEMAP_URL, headers=headers_req)

root = ET.fromstring(response.content)
namespace = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}

raw_links = [elem.text for elem in root.findall('ns:url/ns:loc', namespace)]

# সাধারণ শপ/কার্ট পেজ বাদ দিয়ে কেবল আসল প্রোডাক্ট লিংক ফিল্টার করা
product_links = [link for link in raw_links if link.strip('/').split('/')[-1] not in ['shop', 'cart', 'checkout', 'my-account']]

print(f"ফিল্টার করার পর মোট {len(product_links)} টি আসল প্রোডাক্ট পাওয়া গেছে। স্ক্র্যাপিং শুরু হচ্ছে...\n")

# ৪. লুপ ও স্ক্র্যাপিং
for url in product_links:
    slug = url.strip('/').split('/')[-1]
    product_name = slug.replace('-', ' ').title()

    search_query = f"{product_name} price in Bangladesh"
    google_search_url = f"https://html.duckduckgo.com/html/?q={search_query}"

    res = requests.get(google_search_url, headers=headers_req)
    soup = BeautifulSoup(res.text, 'html.parser')

    prices = []
    comp_urls = []

    # ক্লিন কম্পিটিটর ইউআরএল এক্সট্র্যাক্ট করা
    for a_tag in soup.find_all('a', class_='result__url', limit=15):
        raw_url = a_tag.get('href', '')

        # DuckDuckGo রিডাইরেক্ট ইউআরএল ফিল্টার ও ক্লিন করা
        if 'uddg=' in raw_url:
            clean_url = unquote(raw_url.split('uddg=')[1].split('&')[0])
        else:
            clean_url = raw_url

        if clean_url and "inaayasmart.com" not in clean_url and "duckduckgo.com" not in clean_url:
            if clean_url not in comp_urls:
                comp_urls.append(clean_url)
            if len(comp_urls) == 5:
                break

    # প্রাইস এক্সট্রাকশন (টেক্সট ও স্নিপেট থেকে)
    results_text = soup.get_text()
    found_prices = re.findall(r'(?:Tk|৳|BDT|\b)\s?(\d{2,5})', results_text)

    for p in found_prices:
        val = int(p)
        # ফিল্টারিং logic: খুব ছোট বা অস্বাভাবিক সংখ্যা বাদ দিয়ে প্রাইস এক্সট্রাক্ট
        if 100 <= val <= 200000:
            prices.append(val)

    # গড়, সর্বনিম্ন ও সর্বোচ্চ হিসাব
    if prices:
        # এক্সট্রিম বা ভুয়া ছোট সংখ্যা এড়াতে ৩য় থেকে ৪র্থ সর্বোচ্চ প্র্যাকটিক্যাল রেঞ্জ নেওয়া
        min_price = min(prices)
        max_price = max(prices)
        avg_price = round(sum(prices) / len(prices), 2)
    else:
        min_price, max_price, avg_price = "N/A", "N/A", "N/A"

    comp_urls_str = "\n".join(comp_urls) if comp_urls else "No competitors found"

    # শিটে পাঠানো
    row_data = [product_name, url, comp_urls_str, min_price, max_price, avg_price]
    worksheet.append_row(row_data)
    print(f"আপডেট হয়েছে: {product_name} | Min: {min_price} | Max: {max_price} | Avg: {avg_price}")

print("\nকাজ সফলভাবে সম্পন্ন হয়েছে!")

ফিল্টার করার পর মোট 0 টি আসল প্রোডাক্ট পাওয়া গেছে। স্ক্র্যাপিং শুরু হচ্ছে...


কাজ সফলভাবে সম্পন্ন হয়েছে!
